In [ ]:
##统计分类MAE
import json
from collections import defaultdict
import random

def calculate_mae(results_file, output_file):
    """计算每个subtask的MAE"""
    print(f"Loading results from {results_file}...")
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    # 按subtask分组
    subtask_data = defaultdict(lambda: {'predictions': [], 'labels': []})
    
    for item in results:
        if item['predicted_label'] is not None and item['label'] is not None:
            subtask = item['subtask']
            subtask_data[subtask]['predictions'].append(item['predicted_label'])
            subtask_data[subtask]['labels'].append(item['label'])
        elif item['predicted_label'] is None and item['label'] is not None:
            subtask = item['subtask']
            if subtask == 'Readiness':
                subtask_data[subtask]['predictions'].append(random.randint(0, 10))
                subtask_data[subtask]['labels'].append(item['label'])
            else:
                subtask_data[subtask]['predictions'].append(random.randint(1, 5))
                subtask_data[subtask]['labels'].append(item['label'])
    
    # 计算MAE
    mae_results = {}
    for subtask, data in subtask_data.items():
        predictions = data['predictions']
        labels = data['labels']
        
        if len(predictions) > 0:
            mae = sum(abs(p - l) for p, l in zip(predictions, labels)) / len(predictions)
            mae_results[subtask] = {
                'mae': round(mae, 4),
                'num_samples': len(predictions)
            }
        else:
            mae_results[subtask] = {
                'mae': None,
                'num_samples': 0
            }
    
    # 计算总体MAE
    all_predictions = []
    all_labels = []
    for data in subtask_data.values():
        all_predictions.extend(data['predictions'])
        all_labels.extend(data['labels'])
    
    if len(all_predictions) > 0:
        overall_mae = sum(abs(p - l) for p, l in zip(all_predictions, all_labels)) / len(all_predictions)
        mae_results['overall'] = {
            'mae': round(overall_mae, 4),
            'num_samples': len(all_predictions)
        }
    
    # 保存结果
    print(f"Saving MAE results to {output_file}...")
    with open(output_file, 'w') as f:
        json.dump(mae_results, f, indent=2, ensure_ascii=False)
    
    # 打印结果
    print("\n" + "="*50)
    print("MAE Results by Subtask:")
    print("="*50)
    for subtask, metrics in sorted(mae_results.items()):
        if subtask != 'overall':
            print(f"{subtask:12s}: MAE = {metrics['mae']:.4f} (n={metrics['num_samples']})")
    print("-"*50)
    if 'overall' in mae_results:
        print(f"{'Overall':12s}: MAE = {mae_results['overall']['mae']:.4f} (n={mae_results['overall']['num_samples']})")
    print("="*50)

results_file = '/path/to/inference_results.json'
output_file = '/path/to/mae_results.json'
calculate_mae(results_file, output_file)

Loading results from Causal_RL/eval/causal_edge_sensitivity/global_step_80/test_newprompt_causal_remove_p20_results.json...
Saving MAE results to Causal_RL/eval/causal_edge_sensitivity/global_step_80/remove_p20_mae_results.json...

MAE Results by Subtask:
Fatigue     : MAE = 0.4788 (n=165)
Mood        : MAE = 0.4424 (n=165)
Readiness   : MAE = 1.6258 (n=155)
Stress      : MAE = 0.4516 (n=155)
--------------------------------------------------
Overall     : MAE = 0.7406 (n=640)


In [ ]:
##统计推理质量得分
import json
from collections import defaultdict
def analyze_by_subtask(eval_file, output_file=None):
    """
    按照Subtask统计评测结果的平均分
    
    参数:
        eval_file: 评测结果JSON文件路径
        output_file: 输出统计结果的JSON文件路径（可选）
    """
    print(f"Loading evaluation results from {eval_file}...")
    with open(eval_file, 'r') as f:
        results = json.load(f)
    
    print(f"Total samples: {len(results)}")
    
    # 按subtask分组统计
    subtask_stats = defaultdict(lambda: {
        'readability': [],
        'logical_consistency': [],
        'comprehensiveness': [],
        'count': 0
    })
    
    dimensions = ['readability', 'logical_consistency', 'comprehensiveness']
    
    for item in results:
        subtask = item.get('subtask', 'Unknown')
        evaluations = item.get('evaluations', {})
        
        for dimension in dimensions:
            if dimension in evaluations:
                score = evaluations[dimension].get('score')
                if score is not None:
                    subtask_stats[subtask][dimension].append(score)
        
        subtask_stats[subtask]['count'] += 1
    
    # 计算平均分
    summary = {}
    for subtask, stats in subtask_stats.items():
        summary[subtask] = {
            'count': stats['count']
        }
        
        for dimension in dimensions:
            scores = stats[dimension]
            if scores:
                avg_score = sum(scores) / len(scores)
                summary[subtask][dimension] = {
                    'average': round(avg_score, 3),
                    'num_valid': len(scores)
                }
            else:
                summary[subtask][dimension] = {
                    'average': None,
                    'num_valid': 0
                }
    
    # 计算总体平均分
    overall_stats = {
        'readability': [],
        'logical_consistency': [],
        'comprehensiveness': []
    }
    
    for item in results:
        evaluations = item.get('evaluations', {})
        for dimension in dimensions:
            if dimension in evaluations:
                score = evaluations[dimension].get('score')
                if score is not None:
                    overall_stats[dimension].append(score)
    
    summary['Overall'] = {}
    for dimension in dimensions:
        scores = overall_stats[dimension]
        if scores:
            avg_score = sum(scores) / len(scores)
            summary['Overall'][dimension] = {
                'average': round(avg_score, 3),
                'num_valid': len(scores)
            }
        else:
            summary['Overall'][dimension] = {
                'average': None,
                'num_valid': 0
            }
    
    summary['Overall']['count'] = len(results)
    
    # 保存结果
    if output_file:
        print(f"\nSaving summary to {output_file}...")
        with open(output_file, 'w') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
    
    # 打印结果
    print("\n" + "="*80)
    print("Evaluation Results by Subtask")
    print("="*80)
    print(f"{'Subtask':<15} {'Count':<8} {'Readability':<15} {'Logical Cons.':<15} {'Comprehensive':<15}")
    print("-"*80)
    
    for subtask in sorted(summary.keys()):
        if subtask == 'Overall':
            continue
        
        stats = summary[subtask]
        count = stats['count']
        
        read_avg = stats['readability']['average']
        read_str = f"{read_avg:.3f}" if read_avg is not None else "N/A"
        
        logic_avg = stats['logical_consistency']['average']
        logic_str = f"{logic_avg:.3f}" if logic_avg is not None else "N/A"
        
        comp_avg = stats['comprehensiveness']['average']
        comp_str = f"{comp_avg:.3f}" if comp_avg is not None else "N/A"
        
        print(f"{subtask:<15} {count:<8} {read_str:<15} {logic_str:<15} {comp_str:<15}")
    
    print("-"*80)
    
    # 打印总体统计
    if 'Overall' in summary:
        stats = summary['Overall']
        count = stats['count']
        
        read_avg = stats['readability']['average']
        read_str = f"{read_avg:.3f}" if read_avg is not None else "N/A"
        
        logic_avg = stats['logical_consistency']['average']
        logic_str = f"{logic_avg:.3f}" if logic_avg is not None else "N/A"
        
        comp_avg = stats['comprehensiveness']['average']
        comp_str = f"{comp_avg:.3f}" if comp_avg is not None else "N/A"
        
        print(f"{'Overall':<15} {count:<8} {read_str:<15} {logic_str:<15} {comp_str:<15}")
    
    print("="*80)
    
    return summary



eval_file = '/path/to/eval_results.json'
output_file = '/path/to/eval_summary.json'
    
analyze_by_subtask(eval_file, output_file)

Loading evaluation results from Causal_RL/eval/PMData/causal_edge_sensitivity/test_newprompt_causal_add_p10_reason_eval_results.json...
Total samples: 640

Saving summary to Causal_RL/eval/PMData/causal_edge_sensitivity/test_newprompt_causal_add_p10_reason_eval_summary.json...

Evaluation Results by Subtask
Subtask         Count    Readability     Logical Cons.   Comprehensive  
--------------------------------------------------------------------------------
Fatigue         165      N/A             2.927           3.000          
Mood            165      N/A             2.891           3.000          
Readiness       155      N/A             2.826           3.000          
Stress          155      N/A             2.942           3.000          
--------------------------------------------------------------------------------
Overall         640      N/A             2.897           3.000          


{'Readiness': {'count': 155,
  'readability': {'average': None, 'num_valid': 0},
  'logical_consistency': {'average': 2.826, 'num_valid': 155},
  'comprehensiveness': {'average': 3.0, 'num_valid': 155}},
 'Stress': {'count': 155,
  'readability': {'average': None, 'num_valid': 0},
  'logical_consistency': {'average': 2.942, 'num_valid': 155},
  'comprehensiveness': {'average': 3.0, 'num_valid': 155}},
 'Mood': {'count': 165,
  'readability': {'average': None, 'num_valid': 0},
  'logical_consistency': {'average': 2.891, 'num_valid': 165},
  'comprehensiveness': {'average': 3.0, 'num_valid': 165}},
 'Fatigue': {'count': 165,
  'readability': {'average': None, 'num_valid': 0},
  'logical_consistency': {'average': 2.927, 'num_valid': 165},
  'comprehensiveness': {'average': 3.0, 'num_valid': 165}},
 'Overall': {'readability': {'average': None, 'num_valid': 0},
  'logical_consistency': {'average': 2.897, 'num_valid': 640},
  'comprehensiveness': {'average': 3.0, 'num_valid': 640},
  'count'